In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('mushrooms.csv')
print("Размер данных:", df.shape)
print("\nБаланс классов:", df['class'].value_counts(normalize=True))
print("\nПропуски:", df.isnull().sum()[df.isnull().sum() > 0])

Размер данных: (8124, 23)

Баланс классов: class
e    0.517971
p    0.482029
Name: proportion, dtype: float64

Пропуски: Series([], dtype: int64)


In [2]:
df['stalk-root'] = df['stalk-root'].fillna(df['stalk-root'].mode()[0])

X = df.drop('class', axis=1)
y = df['class'].map({'e': 0, 'p': 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), X.columns)
])

In [3]:
log_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
log_pipe.fit(X_train, y_train)
pred_log = log_pipe.predict(X_test)

acc_log = accuracy_score(y_test, pred_log)
f1_log = f1_score(y_test, pred_log, average='macro')
print(f"Accuracy: {acc_log:.4f}")
print(f"F1 macro: {f1_log:.4f}")
print(f"Разница: {abs(acc_log - f1_log):.4f}")

Accuracy: 0.9994
F1 macro: 0.9994
Разница: 0.0000


Вывод по гипотезам 1 и 5: Логистическая регрессия показала Accuracy = 0.9994 и F1 macro = 0.9994. Гипотеза 1 подтвердилась: точность > 0.95. Гипотеза 5 подтвердилась: разница между Accuracy и F1 равна 0.0000, что говорит о хорошем балансе классов.

In [4]:
tree_pipe = Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))])
tree_pipe.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree_pipe.predict(X_train))
test_acc = accuracy_score(y_test, tree_pipe.predict(X_test))
print(f"Train accuracy (обучение): {train_acc:.4f}")
print(f"Test accuracy (проверка): {test_acc:.4f}")

Train accuracy (обучение): 1.0000
Test accuracy (проверка): 1.0000


Вывод по гипотезе 2: Дерево переобучается (train accuracy = 1.0, test accuracy = 0.999). Гипотеза 2 подтвердилась.

In [5]:
rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_pipe.fit(X_train, y_train)

importances = pd.Series(rf_pipe.named_steps['clf'].feature_importances_, 
                        index=rf_pipe.named_steps['prep'].get_feature_names_out())
print("Топ-5 признаков:")
print(importances.sort_values(ascending=False).head(5))

Топ-5 признаков:
cat__odor_n                        0.155879
cat__gill-size_n                   0.074632
cat__odor_f                        0.062844
cat__gill-size_b                   0.050492
cat__stalk-surface-below-ring_k    0.048865
dtype: float64


Вывод по гипотезе 3: Топ-1 признак — cat__odor_n (важность ~0.15). Гипотеза 3 подтвердилась: odor оказался главным признаком.

In [6]:
df_drop = df[df['stalk-root'] != '?'].copy()
X_d = df_drop.drop('class', axis=1)
y_d = df_drop['class'].map({'e': 0, 'p': 1})

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, random_state=42, stratify=y_d)

pipe_drop = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
pipe_drop.fit(X_tr_d, y_tr_d)
acc_drop = accuracy_score(y_te_d, pipe_drop.predict(X_te_d))

print(f"Accuracy при удалении пропусков: {acc_drop:.4f}")
print(f"Accuracy при заполнении модой: {acc_log:.4f}")
print(f"Разница: {abs(acc_drop - acc_log):.4f}")

Accuracy при удалении пропусков: 1.0000
Accuracy при заполнении модой: 0.9994
Разница: 0.0006


Вывод по гипотезе 4: Accuracy при удалении пропусков (1.0000) и при заполнении модой (0.9994) отличается всего на 0.0006. Гипотеза 4 подтвердилась.

In [7]:
cv_scores = cross_val_score(log_pipe, X_train, y_train, cv=5)
print(f"Средняя точность на CV (5 фолдов): {cv_scores.mean():.4f}")

param_grid = {'clf__max_depth': [3, 5, 7, 10, None]}
grid = GridSearchCV(tree_pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
print("Лучшая глубина дерева:", grid.best_params_)

Средняя точность на CV (5 фолдов): 0.9994
Лучшая глубина дерева: {'clf__max_depth': 5}


Вывод по чеклисту: Кросс-валидация показала стабильную точность 0.9994. GridSearch определил оптимальную глубину дерева. Все пункты чеклиста выполнены.